# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Sep 15, 2026 - Day 7: MLE in practice - real fits, real problems

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">


## Today's plan

* Recap: from week 1's bare formulae, through Bayes' rule, the likelihood, and robust diagnostics, to what maximizing $\mathcal{L}$ still can't do for you
* When there's no closed-form answer: numerical optimization
* A real worked fit: the period-luminosity relation for Cepheids
* Local minima, and why the multimodal-likelihood demo from Day 5/6 was a warning, not a curiosity
* Practice oral defense (ungraded, ~10 min): I defend the Day 1 demo, then volunteers try asking
* EXTRA (time permitting, otherwise in the notes): the $\kappa$-mechanism, and profile likelihoods


In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
# Imports, and the real Cepheid period-luminosity sample (not synthetic).
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Ten real, well-studied Galactic Cepheids. Periods come from a Welch-windowed periodogram run on
# decades of AAVSO visual light-curve photometry (the periodogram technique itself is covered
# formally in the Oct 20/22 time-series unit -- for today, take the periods as real numbers handed
# to you, the same way you'd get them from a collaborator). Absolute V magnitudes are parallax-
# calibrated Galactic Cepheid values, from the same kind of HST Fine Guidance Sensor astrometric
# program that anchors the modern Cepheid distance scale.
with open('data/aavso_cepheids_pl.csv') as f:
    cepheids = list(csv.DictReader(f))

star_names = [c['star'] for c in cepheids]
logP = np.array([np.log10(float(c['period_days'])) for c in cepheids])
M_obs = np.array([float(c['M_V']) for c in cepheids])
N_cep = len(cepheids)

# A single representative photometric/calibration precision, not a per-star measured value: AAVSO
# visual estimates from experienced observers are typically good to a few hundredths of a magnitude,
# and the HST parallax-based zero point carries a comparable systematic.
sigma_obs = np.full(N_cep, 0.06)

# For comparison: Benedict et al. 2007's own published V-band Galactic Cepheid PL relation
# (their Table 12, M_V = a + b*(logP-1) with a=-4.05+/-0.02, b=-2.43+/-0.12 at logP=1;
# converted here to the M = a_lit*logP + b_lit form used throughout this notebook).
a_lit, b_lit = -2.43, -1.62

# A real literature point cloud for comparison: ten Galactic Cepheids with HST Fine Guidance
# Sensor parallax-based absolute magnitudes (Benedict et al. 2007, AJ 133, 1810, Tables 2 & 11).
# delta Cep is in both this sample and our own ten -- same star, same published M_V (-3.47),
# a real cross-check, not a coincidence.
with open('data/benedict2007_cepheids.csv') as f:
    _benedict_rows = list(csv.DictReader(f))
benedict_logP = np.array([np.log10(float(r['period_days'])) for r in _benedict_rows])
benedict_MV = np.array([float(r['M_V']) for r in _benedict_rows])
benedict_names = [r['star'] for r in _benedict_rows]
print(f"{N_cep} real Cepheids, periods {10**logP.min():.1f} to {10**logP.max():.1f} days")


## Recap: the formulae, and where they came from

* week 1 gave you formulae with no derivation: $\bar x$, $s$, IQR - not even why $s^2$ divides by $N-1$
* week 2's probability axioms derive Bayes' rule (posterior $\propto$ likelihood $\times$ prior). Day 5/6 maximized the likelihood half: the sample mean is the MLE that falls out of it
* built on two assumptions doing a lot of quiet work: independent measurements, and Gaussian noise with one shared $\sigma$


In [ ]:
# Visual reminder: mean, s, and IQR on a histogram (a "skip" cell: runs, not a slide)
def plot_recap_hist():
    rng3 = np.random.default_rng(1)
    x = rng3.normal(5, 1.5, 200)
    xbar = x.mean()
    s = x.std(ddof=1)
    q1, q3 = np.percentile(x, [25, 75])
    fig, ax = plt.subplots(figsize=(6, 2.6))
    ax.hist(x, bins=20, color='C0', alpha=0.7)
    ax.axvline(xbar, color='C3', lw=2, label=fr'$\bar x$ = {xbar:.2f}')
    ax.axvspan(xbar - s, xbar + s, color='C3', alpha=0.15, label=fr'$\pm s$ = {s:.2f}')
    ax.axvline(q1, color='C2', ls='--', label='IQR')
    ax.axvline(q3, color='C2', ls='--')
    ax.set_yticks([])
    ax.legend(fontsize=8, loc='upper right', ncol=3)
    plt.tight_layout()
    plt.show()


In [ ]:
# where the mean, s, and IQR actually sit on real-looking data
plot_recap_hist()


## Recap: robust likelihoods, and the diagnostics

A second week-2 idea belongs here - the other place a likelihood choice matters:

* week 2 gave four ways to handle outliers: k-sigma clipping, MAD, mixture models, Theil-Sen/RANSAC
* the mixture model is a **likelihood-level** fix: a better $\mathcal{L}$ for outliers, not an estimator patched after the fact
* the **QQ-plot** is the diagnostic for whether Gaussian is a fair assumption, and for finding exactly where - usually the tails - it stops being one


In [ ]:
# Visual reminder: a QQ-plot catching contaminated tails (a "skip" cell: runs, not a slide)
def plot_recap_qq():
    from scipy import stats
    rng4 = np.random.default_rng(2)
    x = np.concatenate([rng4.normal(0, 1, 180), rng4.normal(0, 6, 20)])  # contaminated tails
    fig, ax = plt.subplots(figsize=(6, 2.6))
    (osm, osr), (slope, intercept, r) = stats.probplot(x, dist='norm')
    ax.plot(osm, osr, 'o', ms=3, color='C0', alpha=0.7)
    ax.plot(osm, slope * osm + intercept, 'C3-', lw=1.5)
    ax.set_xlabel('theoretical quantiles')
    ax.set_ylabel('sample quantiles')
    plt.tight_layout()
    plt.show()


In [ ]:
# a QQ-plot catching contaminated tails departing from the line
plot_recap_qq()


## Recap: what the machinery still can't do

* the MLE recipe applies to any likelihood: solve $\left.\frac{\partial \ln\mathcal{L}}{\partial\theta}\right|_{\hat\theta} = 0$ for $\hat\theta$, then read the curvature $\left.\frac{\partial^2 \ln\mathcal{L}}{\partial\theta^2}\right|_{\hat\theta}$ off for the error bar
* nothing we've built checks whether the likelihood, or the model behind it, is the *right* one - that choice is the scientist's, and stays the scientist's
* more pointed than that: even the right likelihood can't forbid a physically impossible answer on its own - $\hat\varpi = -0.3 \pm 0.5$ mas still has no distance
* the fix for that is a **prior**, coming Sep 29. A different failure - no likelihood to write down at all - needs **simulation-based inference** instead, coming Nov 3/5


## Cepheids: why period predicts luminosity

* Cepheids pulsate in a narrow strip of the HR diagram, the **instability strip** - a partial-ionization He$^+$ zone drives the pulsation (the $\kappa$-mechanism - EXTRA below), with a period set by its mean density. Bigger stars pulsate more slowly.
* Henrietta Leavitt worked as one of Harvard College Observatory's "computers," measuring photographic plates by hand. Cataloguing over 1700 Magellanic Cloud variables (1908, 1912), she found brighter ones had longer periods - and every star in one cloud sits at essentially the same distance, so the trend had to be *intrinsic*.
* Turning that into a ruler took an independent distance to nearby **Galactic** Cepheids: HST Fine Guidance Sensor parallax. That calibration is today's fit.
* Once calibrated: period gives $M_V$, apparent magnitude gives distance - the rung Hubble climbed to show the universe is expanding.

<div style="display:flex; align-items:flex-start; gap:1.2em; margin-top:0.6em;">
<video controls width="480" src="media/cepheid_variable_star_animation_eso.mp4" style="border-radius:4px;"></video>
<div style="text-align:center;">
<img src="images/leavitt_portrait.jpg" alt="Portrait photograph of astronomer Henrietta Swan Leavitt" style="max-height:220px; border-radius:4px;">
<div style="font-size:0.75em; color:#666;">Henrietta Swan Leavitt</div>
</div>
</div>

<div style="font-size:0.7em; color:#666;">Left: artist's illustration of a Cepheid-type variable star pulsating (NASA, ESA, M. Kornmesser / ESA-Hubble). Not in the repository - plays from a local file in class; watch it here: https://www.youtube.com/watch?v=sXJBrRmHPj8</div>


## The Harvard "Computers"

<p align="center"><img src="images/harvard_computers_pickering.jpg" alt="Group photo of women astronomical computers at Harvard College Observatory, circa 1891, analyzing photographic plates" style="display:block;margin:0 auto;max-height:480px"></p>


## Leavitt's 1912 discovery plot

<p align="center"><img src="images/leavitt_1912_pl_plot.jpg" alt="Henrietta Leavitt's original 1912 period-luminosity plot for Small Magellanic Cloud Cepheids, Harvard College Observatory Circular 173" style="display:block;margin:0 auto;max-height:480px"></p>


## The instability strip

* every star crossing this strip pulsates by the same mechanism, just at different masses and ages: **RR Lyrae** (old, low-mass, common in globular clusters) at short period, **classical Cepheids** (young, massive) at intermediate period, **W Virginis** stars (evolved, low-mass) at long period for their luminosity
* only the classical Cepheids give the clean period-luminosity relation used here - RR Lyrae and W Virginis stars follow their own, offset relations

<p align="center"><img src="images/cepheid_instability_strip.png" alt="HR diagram with the classical Cepheid instability strip marked, crossing the giant branch between the main sequence and the supergiants" style="display:block;margin:0 auto;max-height:400px"></p>


## From toy Gaussians to a real fit

* most likelihoods so far had a closed-form maximum: differentiate, set to zero, solve
* the Leavitt law would still have one, if we knew the total scatter exactly - we don't
* that turns $\chi^2$ into something with no closed-form solution, so it must be minimized **numerically**

$$M_V(P) = a\,\log_{10} P + b$$

Two parameters, a straight line - but the fitting code doesn't know that. It searches for where $\chi^2(a,b)$ stops getting smaller.


## The Cepheid sample

Ten real Galactic Cepheids, not synthetic data: $\delta$ Cep, $\eta$ Aql, RS Pup, RT Aur, S Sge, SV Vul, U Vul, W Sgr, X Cyg, $\zeta$ Gem. Periods come from up to a century of visual photometry archived by the AAVSO (American Association of Variable Star Observers) - the same kind of period-magnitude diagram Leavitt built by hand for her period-luminosity ("Leavitt") law in 1912.


## A real Cepheid light curve

One star from the sample, up close: $\delta$ Cep itself, the class prototype - a year of nightly visual brightness estimates from AAVSO observers, phased on its real 5.37-day period.

* that period came from a periodogram - if ASTR 310's Fourier analysis unit is still fresh, this is the same idea: find the dominant frequency in unevenly-sampled data
* we formalize periodograms and time-series methods properly in a few weeks (Oct 20/22) - for today, take the period as a real number handed to you


In [ ]:
# Load delta Cep's real AAVSO light curve (one observing season, visual photometry) and phase-fold it
import csv as _csv
with open('data/delcep_lightcurve.csv') as f:
    _rows = list(_csv.DictReader(f))
delcep_t = np.array([float(r['JD']) for r in _rows])
delcep_m = np.array([float(r['Magnitude']) for r in _rows])
delcep_P = 5.366  # real period, days (same value used in the AAVSO sample above)

def plot_delcep_lightcurve():
    phase = (delcep_t / delcep_P) % 1
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(phase, delcep_m, 'k.', ms=3, alpha=0.5)
    ax.plot(phase + 1, delcep_m, 'k.', ms=3, alpha=0.5)
    ax.invert_yaxis()
    ax.set_xlabel('phase')
    ax.set_ylabel('visual magnitude')
    ax.set_title(r'$\delta$ Cep, phased on P = 5.366 d (AAVSO visual estimates, 2019-2020)')
    plt.tight_layout()
    plt.show()


In [ ]:
# delta Cep's real, phase-folded light curve - not an idealized sine wave, real scatter and all
plot_delcep_lightcurve()


In [ ]:
# Plot the real Cepheid sample: absolute magnitude vs period, with error bars
def plot_cepheids(fit_ab=None, lit_ab=None, lit_cloud=None):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.errorbar(10**logP, M_obs, yerr=sigma_obs, fmt='o', ms=6, color='C0', alpha=0.8,
                ecolor='0.6', label='AAVSO Cepheids (ours)')
    for name, p, m in zip(star_names, 10**logP, M_obs):
        ax.annotate(name, (p, m), textcoords='offset points', xytext=(6, 4), fontsize=8)
    if lit_cloud is not None:
        clogP, cMV = lit_cloud
        ax.plot(10**clogP, cMV, '^', ms=6, color='C1', alpha=0.85,
                label='Benedict et al. 2007 (literature)')
    x_line = np.array([logP.min(), logP.max()])
    if fit_ab is not None:
        a, b = fit_ab
        ax.plot(10**x_line, a * x_line + b, '-', color='C3', lw=2, label='MLE fit')
    if lit_ab is not None:
        a, b = lit_ab
        ax.plot(10**x_line, a * x_line + b, '--', color='0.4', lw=1.5, label='literature fit')
    ax.set_xscale('log')
    ax.invert_yaxis()
    ax.set_xlabel('Period [days]')
    ax.set_ylabel(r'$M_V$')
    ax.set_title('10 real Galactic Cepheids (AAVSO periods, parallax-calibrated $M_V$)')
    if fit_ab is not None or lit_ab is not None or lit_cloud is not None:
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# 10 real Cepheids: absolute magnitude vs period, with the representative error bar
plot_cepheids()


* here we use one representative $\sigma_{obs}$ for all ten stars, a simplification - real per-star precision does vary
* what it can't capture is **intrinsic** scatter: metallicity, blending, and imperfect mean-light corrections that make real Cepheids scatter around the line by more than photometric error alone predicts - a second, unmeasured source of noise the fit has to account for, or the error bars on $a$ and $b$ are wrong


## Fitting with an unknown nuisance parameter

The likelihood needs a total variance per point: $\sigma_i^2 = \sigma_{obs,i}^2 + \sigma_{int}^2$. We don't know $\sigma_{int}$ ahead of time, and it's a **nuisance parameter** - not the thing we want, but the fit is wrong without it. (Bayesian treatments marginalize these out - coming Sep 29; here we just fit them alongside everything else.)

$$-2\ln\mathcal{L}(a, b, \sigma_{int}) = \sum_i \left[ \frac{(M_{obs,i} - a\log_{10}P_i - b)^2}{\sigma_{obs,i}^2 + \sigma_{int}^2} + \ln(\sigma_{obs,i}^2 + \sigma_{int}^2) \right]$$

Three parameters now, and the second term (the $\ln$) is there so the fit can't cheat by inflating $\sigma_{int}$ to make every point look consistent - it pays for a wider error bar.


In [ ]:
# Minimize -2 ln L over (a, b, sigma_int) with scipy.optimize; report the curvature-based error bars.
def neg2logL(theta):
    a, b, log_sig_int = theta
    sig_int = np.exp(log_sig_int)          # fit in log(sigma_int) so the optimizer can't wander negative
    var = sigma_obs**2 + sig_int**2
    resid = M_obs - (a * logP + b)
    return np.sum(resid**2 / var + np.log(var))

def fit_cepheids(x0=(-2.0, -1.0, np.log(0.3))):
    res = minimize(neg2logL, x0=x0, method='Nelder-Mead',
                    options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 5000})
    # curvature (Hessian) by finite differences, for the error bars
    h = 1e-4
    n = len(res.x)
    hess = np.zeros((n, n))
    f0 = neg2logL(res.x)
    for i in range(n):
        for j in range(n):
            ei, ej = np.zeros(n), np.zeros(n)
            ei[i] = h; ej[j] = h
            hess[i, j] = (neg2logL(res.x + ei + ej) - neg2logL(res.x + ei - ej)
                          - neg2logL(res.x - ei + ej) + neg2logL(res.x - ei - ej)) / (4 * h**2)
    cov = 2 * np.linalg.inv(hess)   # factor of 2: -2lnL is chi^2, not -lnL
    errs = np.sqrt(np.diag(cov))
    return res, errs


In [ ]:
# fit the Leavitt law (a, b) plus an unknown intrinsic scatter, numerically
res, errs = fit_cepheids()
a_hat, b_hat, sig_int_hat = res.x[0], res.x[1], np.exp(res.x[2])
print(f"a = {a_hat:.3f} +/- {errs[0]:.3f}   (literature: {a_lit})")
print(f"b = {b_hat:.3f} +/- {errs[1]:.3f}   (literature: {b_lit})")
print(f"sigma_int = {sig_int_hat:.3f} mag")
print(f"slope vs. literature: {(a_hat - a_lit)/errs[0]:+.1f} sigma")
print(f"intercept vs. literature: {(b_hat - b_lit)/errs[1]:+.1f} sigma")


In [ ]:
# Same data, now with the fitted line, the literature line, and a real literature point cloud on top
plot_cepheids(fit_ab=(a_hat, b_hat), lit_ab=(a_lit, b_lit), lit_cloud=(benedict_logP, benedict_MV))


* $a$ and $b$ each come back well under 1$\sigma$ of Benedict et al. (2007)'s own published Galactic Cepheid period-luminosity relation - an independent HST-parallax calibration of ten other Cepheids: a century of amateur AAVSO visual photometry and a Nelder-Mead optimizer reproduce it
* $\sigma_{int}\approx0.06$ mag came back nonzero: real Cepheids scatter around the line by more than photometric precision alone predicts
* this took an optimizer: $\sigma_{int}$ is fit as $\log\sigma_{int}$ so it can't wander negative, and `scipy.optimize.minimize(method='Nelder-Mead')` never uses a derivative - it never hands you an automatic `hess_inv` the way Day 5/6's demo did, so the printed $\pm$ here comes from a Hessian built explicitly, by finite differences


## Optimizers find *a* peak, not *the* peak

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A numerical optimizer reports where it stopped. That is not proof no lower valley exists elsewhere.**

</div>

Recall the six-night sinusoid from Day 5/6: the likelihood over (period, phase) had multiple islands from aliasing, each one a **local minimum** of $-2\ln\mathcal{L}$ - and Day 5/6 showed the true period isn't even the tallest island, so picking the best-looking peak doesn't save you either. Minimize from a bad starting guess and you end up on the wrong island with a perfectly confident, perfectly wrong error bar. Here it is again:


In [ ]:
# Six nightly observations of a sinusoid with unknown period and phase, same demo as Day 5/6.
# Recalled here so the "real multimodal problem" picture is on screen, not just in memory.
def demo_period_alias(P_true=2.3, phi_true=0.7, n_epochs=6, sig=0.15, seed=11):
    rng2 = np.random.default_rng(seed)
    t = np.sort(np.arange(n_epochs) * 1.0 + rng2.uniform(-0.05, 0.05, n_epochs))
    y = np.sin(2*np.pi*t/P_true + phi_true) + rng2.normal(0, sig, n_epochs)
    P = np.linspace(0.6, 6.0, 1200)
    phi = np.linspace(0, 2*np.pi, 240)
    PP, FF = np.meshgrid(P, phi, indexing='ij')
    chi2 = ((y[None, None, :] - np.sin(2*np.pi*t[None, None, :]/PP[..., None] + FF[..., None]))**2).sum(-1) / sig**2
    L = np.exp(-0.5 * (chi2 - chi2.min()))
    fig = plt.figure(figsize=(9, 7))
    gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[1, 3], hspace=0.05, wspace=0.05)
    ax_top = fig.add_subplot(gs[0, 0])
    ax_main = fig.add_subplot(gs[1, 0])
    ax_right = fig.add_subplot(gs[1, 1])
    ax_main.contourf(P, phi, L.T, levels=20, cmap='Greys')
    ax_main.set_xlabel('period P [d]')
    ax_main.set_ylabel('phase [rad]')
    ax_main.axvline(P_true, color='C2', ls=':', label='true period')
    ax_main.legend(loc='upper right')
    # zoomed to where both alias peaks actually sit, so they aren't lost in empty margin
    ax_main.set_xlim(0, 3)
    ax_main.set_ylim(0, 3)
    ax_top.plot(P, L.sum(1) / L.sum(1).max(), color='C3')
    ax_top.set_xlim(0, 3)
    ax_top.set_xticks([])
    ax_top.set_ylabel('L(P)')
    ax_right.plot(L.sum(0) / L.sum(0).max(), phi, color='C3')
    ax_right.set_ylim(0, 3)
    ax_right.set_yticks([])
    ax_right.set_xlabel('L(phase)')
    plt.show()


In [ ]:
# the corner plot from Day 5/6, recalled here rather than left to memory
demo_period_alias()


* the aliases sit at $1/(1/P \pm 1\ \text{cycle/day})$ - the nightly cadence, not a coincidence
* trying several starting points (a **multistart** check) does not reliably find the true island here - the fix for aliasing like this is more data at a different spacing, or sampling the full likelihood (MCMC, in a few weeks), not a better optimizer


In [ ]:
# Same Cepheid fit, but started from three different initial guesses for the slope.
def multistart_demo():
    starts = [-1.0, -2.45, -6.0]
    fig, ax = plt.subplots(figsize=(7, 4))
    a_grid = np.linspace(-8, 2, 400)
    for a0 in starts:
        res_i = minimize(lambda th: neg2logL([th[0], b_hat, np.log(sig_int_hat)]), x0=[a0],
                          method='Nelder-Mead')
        ax.axvline(res_i.x[0], ls='--', label=f'start a0={a0}: converged a={res_i.x[0]:.2f}')
    nll = [neg2logL([a, b_hat, np.log(sig_int_hat)]) for a in a_grid]
    ax.plot(a_grid, nll, color='0.3')
    ax.set_xlabel('slope a')
    ax.set_ylabel(r'$-2\ln\mathcal{L}$ (b, $\sigma_{int}$ fixed at best fit)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# for THIS well-behaved linear model, all three starting guesses converge to the same slope
multistart_demo()


* for a straight line, $-2\ln\mathcal{L}$ really is convex in $a$ (one bowl-shaped minimum, no separate local dips) - so every start ends up in the same place, and this **multistart** check comes up empty on purpose: this demo is meant to look boring
* it is boring because the model is simple; the corner plot above is what a real multimodal problem looks like, with truly separate local minima (the islands) from period aliasing
* the Cepheid demo above didn't need a fix. Some multimodal problems do respond to trying several starting points - but not the sinusoid's: its aliasing comes from the observing cadence itself, so no optimizer fixes it (see the note just after the corner plot above)


## Practice oral defense (ungraded)

Same format as every real defense: 7 minutes, four question slots - opener, a choice you made, a check you ran, one step further. No trick questions.

* I defend [Day 1's AI demo notebook](https://github.com/gnarayan/ast457_2026_Fall/blob/main/lecture/01/day1_ai_course_intro.ipynb) - not your own work, mine, so there's nothing on the line yet
* A couple of volunteers each take a turn asking me a question from one of the four slots - practicing the questioner's chair, not the defense
* Nothing here is graded. This is about seeing the format before anything counts
* Graded Lab 01 defenses start after today's wrap-up - same four slots, same format as Day 1's "What an oral defense sounds like" slide


## Wrapping up MLE in practice

* most real models have no closed-form MLE: you minimize $-2\ln\mathcal{L}$ numerically, same idea as $\chi^2$ minimization, just with a computer doing the algebra
* nuisance parameters (like $\sigma_{int}$) have to be fit even when you don't care about their value, or your other error bars are wrong
* an optimizer reports where it stopped, and that's all you get: for an actually multimodal problem like the aliasing case, trying more starting points can still miss the true answer - sampling the full likelihood is the real fix
* reporting an error bar on one parameter while others are uncertain too takes one more idea - profile likelihoods, EXTRA below if we have time, otherwise in the notes
* next time: is the model even right, and how do you compare two of them?


## EXTRA: the $\kappa$-mechanism

* opacity $\kappa$ normally drops as gas compresses and heats: more compression makes it more transparent, so any compression just drains back out - stable
* inside the instability strip, a layer of partially-ionized helium (He$^+$ $\to$ He$^{2+}$) does the opposite: compressing it ionizes more of the gas, and a partially-ionized gas is MORE opaque, not less
* that traps heat, builds pressure, and pushes the star back out - a self-sustaining valve that drives the pulsation instead of damping it
* only stars whose He$^+$ layer sits at the right depth (the instability strip) pulsate this way - too hot or too cool and the layer is in the wrong place to drive anything


## EXTRA: Profile likelihoods - parameters you don't care about

$\sigma_{int}$ mattered for getting $a$ and $b$ right, but we rarely report it. A **profile likelihood** maximizes over the nuisance parameter at each value of the one you care about:

$$\mathcal{L}_{profile}(a) = \max_{b,\, \sigma_{int}} \mathcal{L}(a, b, \sigma_{int})$$


In [ ]:
# Profile likelihood over the slope a: at each a, re-optimize (b, sigma_int); compare to holding them fixed.
def profile_demo():
    a_grid = np.linspace(-3.6, -2.0, 25)
    profiled, fixed = [], []
    for a in a_grid:
        r = minimize(lambda p: neg2logL([a, p[0], p[1]]), x0=[b_hat, np.log(sig_int_hat)],
                     method='Nelder-Mead')
        profiled.append(r.fun)
        fixed.append(neg2logL([a, b_hat, np.log(sig_int_hat)]))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(a_grid, np.array(profiled) - min(profiled), label='profiled over b, sigma_int', color='C3')
    ax.plot(a_grid, np.array(fixed) - min(fixed), '--', label='b, sigma_int held at best-fit', color='C0')
    ax.axhline(1.0, color='0.6', ls=':', label=r'$\Delta(-2\ln\mathcal{L})=1$')
    ax.set_xlabel('slope a')
    ax.set_ylabel(r'$-2\ln\mathcal{L} - \min$')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# profiled vs. naively-fixed nuisance parameters: the error bar on the slope changes
profile_demo()


* letting $b$ and $\sigma_{int}$ readjust as $a$ moves away from its best fit finds some combination that fits almost as well - that flexibility is real, and an error bar that ignores it is too small
* this is the honest way to report "the slope, profiling over how well we know the intercept and the scatter" - profile likelihood *maximizes* over the nuisance parameters at each $a$, it doesn't integrate them out (that's a different, Bayesian operation)
* the dashed line at $\Delta(-2\ln\mathcal{L})=1$ is the same one-sigma rule from Day 5/6 - one degree of freedom, because only the one parameter you're profiling over ($a$) is held fixed at each grid point while everything else readjusts freely


## Before you go

* Lab 03 status: check the live assignment tracker for what's due this week.
* Nothing from today is collected.
* Colloquium next week (Sep 22): Charlotte Ward (PSU), "Better Together: Combining the Strengths of Rubin, Euclid, and Roman for Time-domain Science and Cosmology" - picks up today's time-domain/distance-ladder thread
* Thursday: model comparison - is a second component even justified by the data?
